[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/16_webscraping/48_scraping_with_firecrawl.ipynb)

# 📓 Notebook 48 — Scraping with Firecrawl

> **Module:** Web Scraping (Module 16) · **Estimated time:** ~60 minutes · **Difficulty:** Intermediate

In **NB 47** you learned to scrape by hand: `requests` fetches the HTML, **BeautifulSoup** picks it apart, and you stay polite with `robots.txt`, throttling, and caching. That stack is perfect for **static, well-structured pages** — and it's free.

But the modern web fights back. The moment DIY scraping hits **JavaScript rendering** (the content is drawn by a browser, so `requests` gets an empty shell), **anti-bot defences** (Cloudflare challenges, fingerprinting, IP blocks), or just **messy HTML** you have to strip down to clean text, you're suddenly maintaining a headless-browser + proxy + boilerplate-cleaning pipeline. That's a lot of undifferentiated plumbing to build and babysit.

This is where a **managed scraping API** earns its keep. **[Firecrawl](https://firecrawl.dev)** takes a URL and hands you back **LLM-ready markdown** or **typed JSON** — it runs the browser, solves the anti-bot dance, and strips the nav bars, ads, and cookie banners for you. One call, clean content.

> 🧭 **Mental model — "give me a URL, get back LLM-ready content."** Firecrawl is not a browser you drive; it's an endpoint you ask. Four verbs cover almost everything:
> - **scrape** one page → markdown / html / links / a screenshot / typed JSON
> - **crawl** a whole site → follow links, return content for every page
> - **map** a site → just the list of its URLs, fast (great for planning a crawl)
> - **search** the web → find pages *and* scrape them in one call
>
> Everything below runs against an offline **`MockFirecrawl`** — no API key, no network — so you can learn the shape of the API deterministically. Swap in the real client (shown in §2) and the exact same code hits the live service.

> 📎 This is the **dedicated** Firecrawl lesson. For a lightning intro that shows Firecrawl next to the DIY BeautifulSoup stack in one place, see the appendix `../03_real_world_io/A1_web_scraping_firecrawl.ipynb`.

## ✅ Prerequisites

- **NB 47 — Web Scraping Fundamentals** (this module): `requests` + BeautifulSoup, `robots.txt`, the fetch → parse → extract → store pipeline, and *why* hand-rolled scraping starts to hurt. We pick up exactly where its pain points end.
- **NB 25 — Document Processing** (Module 6) is helpful: the same structured-extraction mindset (schema in, typed records out) that we apply to scraped pages here.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Explain **when a managed scraping API beats DIY** `requests` + BeautifulSoup — and when it's overkill.
2. Bootstrap a Firecrawl client that uses the **real SDK when installed** (with `FIRECRAWL_API_KEY`) and **falls back to an offline mock** otherwise.
3. **Scrape** a single URL into clean, LLM-ready **markdown** plus its **metadata**.
4. **Extract typed JSON** with a **schema** (a `pydantic` model, or a `dataclass` fallback) and validate it at the boundary.
5. **Crawl** a site, **map** its URLs, and **search**-and-scrape — reading each response's shape correctly.
6. **Batch** many URLs into a tidy **DataFrame** ready for RAG or SQL.
7. Reason about **cost, rate limits, and when *not*** to reach for a managed API.

## 1. Why managed scraping — the DIY pain points, recapped

NB 47 ended on an honest note: `requests` + BeautifulSoup is great until the page stops cooperating. Here is that list of failure modes again — the ones that push you from "I'll scrape it myself" toward "let a service do the hard part":

| DIY pain point (from NB 47) | Why `requests` + `bs4` struggles | What a managed API does instead |
|---|---|---|
| **JavaScript rendering** | `requests` fetches the empty shell; the content is drawn later by JS in a real browser | runs a headless browser for you, returns the *rendered* page |
| **Anti-bot / Cloudflare** | challenges, fingerprinting, and IP blocks stop a naive client | rotating proxies + stealth browsers, handled server-side |
| **Layout drift** | CSS selectors break on every redesign — silent `None`s and `AttributeError`s | LLM extraction against a **schema**, not brittle selectors |
| **Crawling & pagination** | you hand-write the link-following loop, dedup, and depth limits | one `crawl()` call maps and follows the whole site |
| **"Just give me clean text"** | HTML is full of nav, ads, and cookie-banner noise | returns stripped-down **markdown** an LLM can read directly |

> 🧠 **Why markdown is the whole point.** An LLM does not want `<div class="nav-wrapper">…</div>` — it wants the *content*. Firecrawl returns the page as the markdown a human would read, which drops straight into a RAG chunker (Module 6, NB 23) or an agent's context (Module 8) with **zero HTML-cleanup code on your side**. "Scrape → markdown → embed" is the modern ingestion pipeline, and it's why these services exist.

You *can* solve each row by hand (Playwright for JS, proxies for blocks, a custom crawler…). The question this lesson answers is: **when is it worth paying an API to make all five rows disappear at once?**

## 2. Setup — the real client, with an offline mock fallback

The real Firecrawl SDK is one install and one API key:

```bash
pip install firecrawl-py         # the Python SDK — note it imports as `firecrawl`
export FIRECRAWL_API_KEY=fc-...  # get a key at https://firecrawl.dev
```

```python
from firecrawl import Firecrawl
app = Firecrawl(api_key="fc-...")           # or reads FIRECRAWL_API_KEY from the environment
doc = app.scrape("https://firecrawl.dev", formats=["markdown"])
print(doc.markdown)                          # clean, LLM-ready markdown
```

So this notebook needs **no key and no network**, we mirror the small slice of the v2 SDK surface we use — `scrape` / `crawl` / `map` / `search`, returning a document object with `.markdown`, `.json`, `.links`, and `.metadata` — in a **`MockFirecrawl`**. The bootstrap in the third cell below tries the real SDK first and only falls back to the mock, so **the same lesson code runs live the instant you install the SDK and set a key.**

First, the page our mock pretends to scrape — a deliberately noisy HTML fixture (nav, cookie banner, `<script>` tags), the same *fetch* input NB 47 handed to BeautifulSoup:

In [1]:
# A local HTML fixture — the noisy page a browser would load. In real life this is
# whatever `requests.get(url).text` (NB 47) or Firecrawl's headless browser fetches.
# Firecrawl's job is to throw the nav/ads/scripts away and keep the *content*.
HTML_FIXTURE = """
<html><head><title>The Polite Bookshop — Catalogue (page 1)</title></head>
<body>
  <nav class="site-nav">Home · Basket · Account · <a href="/login">Sign in</a></nav>
  <div class="cookie-banner">We use cookies. <button>Accept all</button></div>
  <script>window.__ANALYTICS__ = true;  // tracking noise an LLM does not want</script>
  <main>
    <h1>Catalogue — page 1</h1>
    <article class="product"><h3>Clean Code</h3><span class="price">£32.50</span>
      <span class="rating" data-stars="4">In stock (12)</span></article>
    <article class="product"><h3>The Pragmatic Programmer</h3><span class="price">£28.99</span>
      <span class="rating" data-stars="5">In stock (3)</span></article>
    <article class="product"><h3>Designing Data-Intensive Applications</h3><span class="price">£41.00</span>
      <span class="rating" data-stars="5">Out of stock</span></article>
  </main>
  <a class="next" href="/catalogue/page-2">next →</a>
  <footer>© 2026 The Polite Bookshop · <a href="/privacy">Privacy</a></footer>
</body></html>
"""
print(f"fixture is {len(HTML_FIXTURE)} chars of HTML — nav, cookie banner, and a <script> included.")

fixture is 1038 chars of HTML — nav, cookie banner, and a <script> included.


In [2]:
# ── Offline MockFirecrawl ──────────────────────────────────────────────────
# Mimics the v2 SDK: app.scrape(...) -> a Document with .markdown / .json / .links /
# .metadata ; app.crawl / .map / .search return the job-style dicts the real API does.
# This is the *only* reason the notebook runs with no key and no network.

class MockDocument:
    """Stand-in for a Firecrawl Document."""
    def __init__(self, markdown=None, json=None, links=None, metadata=None, html=None):
        self.markdown = markdown
        self.json = json                       # typed extraction result (dict) when a json format is asked for
        self.links = links or []
        self.metadata = metadata or {}
        self.html = html
    def __repr__(self):
        return f"MockDocument(markdown={self.markdown is not None}, json={self.json is not None}, links={len(self.links)})"


class MockFirecrawl:
    # What Firecrawl would return after stripping the fixture's nav/ads/scripts: clean markdown.
    _MARKDOWN = """# The Polite Bookshop — Catalogue (page 1)

A small, well-behaved demo shop used across the Web Scraping module. Prices in GBP.

## Featured titles

1. **Clean Code** — £32.50 · ★★★★ · In stock (12)
2. **The Pragmatic Programmer** — £28.99 · ★★★★★ · In stock (3)
3. **Designing Data-Intensive Applications** — £41.00 · ★★★★★ · Out of stock

[Next page →](/catalogue/page-2)
"""
    _METADATA = {
        "title": "The Polite Bookshop — Catalogue (page 1)",
        "description": "A small demo bookshop used across the Web Scraping module.",
        "language": "en",
        "statusCode": 200,
    }
    # The records Firecrawl's LLM extraction would fill from the page (as plain dicts).
    _PRODUCTS = [
        {"title": "Clean Code", "price": 32.50, "rating": 4, "in_stock": True},
        {"title": "The Pragmatic Programmer", "price": 28.99, "rating": 5, "in_stock": True},
        {"title": "Designing Data-Intensive Applications", "price": 41.00, "rating": 5, "in_stock": False},
    ]
    # Per-product detail pages (richer than the list view — note the extra `author`).
    _DETAILS = {
        "clean-code": {"title": "Clean Code", "price": 32.50, "rating": 4, "in_stock": True, "author": "Robert C. Martin"},
        "pragmatic-programmer": {"title": "The Pragmatic Programmer", "price": 28.99, "rating": 5, "in_stock": True, "author": "Hunt & Thomas"},
        "ddia": {"title": "Designing Data-Intensive Applications", "price": 41.00, "rating": 5, "in_stock": False, "author": "Martin Kleppmann"},
    }
    # A tiny site for crawl(): url -> clean markdown.
    _SITE_PAGES = {
        "https://shop.example.com/": "# The Polite Bookshop\n\nWelcome. Browse the [catalogue](/catalogue/page-1).",
        "https://shop.example.com/catalogue/page-1": _MARKDOWN,
        "https://shop.example.com/catalogue/page-2": "# Catalogue — page 2\n\n4. **Refactoring** — £34.00 · In stock",
        "https://shop.example.com/about": "# About\n\nA fictional shop for teaching polite scraping.",
    }

    @staticmethod
    def _wants_json(formats):
        return any(isinstance(f, dict) and f.get("type") in ("json", "extract") for f in (formats or []))

    def scrape(self, url, formats=None):
        formats = formats or ["markdown"]
        meta = dict(self._METADATA, sourceURL=url)
        if self._wants_json(formats):
            slug = url.rstrip("/").split("/")[-1]
            if slug in self._DETAILS:                       # a product-detail page -> one typed record
                rec = dict(self._DETAILS[slug])
                return MockDocument(markdown=f"# {rec['title']}\n\n£{rec['price']:.2f}", json={"product": rec}, metadata=meta)
            return MockDocument(markdown=self._MARKDOWN, json={"products": [dict(p) for p in self._PRODUCTS]}, metadata=meta)
        links = [f"https://shop.example.com/products/{s}" for s in self._DETAILS]
        links.append("https://shop.example.com/catalogue/page-2")
        return MockDocument(markdown=self._MARKDOWN, links=links, metadata=meta)

    def crawl(self, url, limit=10, **kw):                   # returns a JOB dict, not a Document
        pages = list(self._SITE_PAGES.items())[:limit]
        data = [MockDocument(markdown=m, metadata={"sourceURL": u, "statusCode": 200}) for u, m in pages]
        return {"status": "completed", "total": len(data), "completed": len(data), "data": data}

    def map(self, url, limit=30, **kw):                     # just the URLs, fast
        base = url.rstrip("/")
        links = [base + p for p in ["/", "/catalogue/page-1", "/catalogue/page-2",
                                    "/products/clean-code", "/products/pragmatic-programmer",
                                    "/products/ddia", "/about"]]
        return {"status": "completed", "links": links[:limit]}

    def search(self, query, limit=5, **kw):                 # web search + scrape the hits
        results = [
            {"url": "https://shop.example.com/products/ddia", "title": "Designing Data-Intensive Applications",
             "markdown": "# Designing Data-Intensive Applications\n\n£41.00 · the go-to systems book."},
            {"url": "https://other.example.com/reviews/ddia", "title": "DDIA — a review",
             "markdown": "# Review: DDIA\n\nStill the reference for data-intensive systems."},
        ]
        return {"query": query, "data": results[:limit]}

print("MockFirecrawl ready — scrape / crawl / map / search, all offline.")

MockFirecrawl ready — scrape / crawl / map / search, all offline.


In [3]:
# Bootstrap the client: prefer the REAL SDK (+ API key), else fall back to the mock.
import os

HAS_FIRECRAWL = False
try:
    from firecrawl import Firecrawl        # pip install firecrawl-py
    HAS_FIRECRAWL = True
except ModuleNotFoundError:
    HAS_FIRECRAWL = False

if HAS_FIRECRAWL and os.getenv("FIRECRAWL_API_KEY"):
    app = Firecrawl(api_key=os.environ["FIRECRAWL_API_KEY"])
    MODE = "REAL Firecrawl (SDK installed + FIRECRAWL_API_KEY found)"
else:
    app = MockFirecrawl()
    reason = "SDK not installed" if not HAS_FIRECRAWL else "no FIRECRAWL_API_KEY set"
    MODE = f"offline MockFirecrawl ({reason}) — everything below runs with no network"

print("Client:", MODE)

Client: offline MockFirecrawl (SDK not installed) — everything below runs with no network


## 3. `scrape` — one URL into clean markdown + metadata

The workhorse call is `scrape(url, formats=[...])`. Ask for `"markdown"` and you get the page as clean, readable text — nav bars, cookie banners, and `<script>` tags already gone. Every document also carries **`.metadata`**: the title, description, language, HTTP status, and source URL. That metadata is gold for a RAG pipeline (it becomes the citation you show the user).

In [4]:
# Scrape a single page into markdown. Compare this to the raw HTML_FIXTURE above:
# the nav, cookie banner, <script>, and footer are gone — only the content remains.
doc = app.scrape("https://shop.example.com/catalogue/page-1", formats=["markdown"])

print(doc.markdown)
print("--- metadata " + "-" * 40)
for k, v in doc.metadata.items():
    print(f"  {k:>12} : {v}")

# The Polite Bookshop — Catalogue (page 1)

A small, well-behaved demo shop used across the Web Scraping module. Prices in GBP.

## Featured titles

1. **Clean Code** — £32.50 · ★★★★ · In stock (12)
2. **The Pragmatic Programmer** — £28.99 · ★★★★★ · In stock (3)
3. **Designing Data-Intensive Applications** — £41.00 · ★★★★★ · Out of stock

[Next page →](/catalogue/page-2)

--- metadata ----------------------------------------
         title : The Polite Bookshop — Catalogue (page 1)
   description : A small demo bookshop used across the Web Scraping module.
      language : en
    statusCode : 200
     sourceURL : https://shop.example.com/catalogue/page-1


> 🔬 **What can `formats` return?** Each entry asks for one representation of the page:
>
> | format | you get back on the document | typical use |
> |---|---|---|
> | `"markdown"` | `.markdown` — clean text | RAG ingestion, summarisation |
> | `"html"` | `.html` — cleaned HTML | when you still want tags |
> | `"links"` | `.links` — every outbound URL | crawling, link analysis |
> | `"screenshot"` | an image URL | visual QA, archiving |
> | `{"type": "json", "schema": ...}` | `.json` — **typed records** | the structured-extraction of §4 |
>
> You can ask for several at once, e.g. `formats=["markdown", "links"]`.

---

### ✋ Quick exercise (~2 min) — Read the metadata

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Right after a scrape you usually log what came in. Reusing the `doc` from §3 (no re-scraping), pull the page's **title** out of `doc.metadata` and count how many **non-empty lines** of clean markdown you got.

In [5]:
# ✍️ Your turn 👇
# Reuse `doc` from §3 — no re-scraping.
page_title = ...     # pull the page's title out of doc.metadata
md_lines   = ...     # count the NON-EMPTY lines of doc.markdown
# print("title:", page_title, "· non-empty markdown lines:", md_lines)   # ← uncomment once filled in

<details>
<summary>✅ <b>Solution</b></summary>

```python
page_title = doc.metadata["title"]
md_lines   = len([ln for ln in doc.markdown.splitlines() if ln.strip()])
print("title:", page_title, "· non-empty markdown lines:", md_lines)
```

`doc.metadata` is a plain dict, so `["title"]` reads the field; splitting the markdown and keeping only lines that survive `.strip()` counts the real content lines. This two-line health check is exactly what you'd log after every ingestion run.
</details>

## 4. Structured extraction — markdown is nice, *typed JSON* is better

Markdown is perfect for "feed the page to an LLM." But when you want **rows in a table** — a price, a rating, a boolean in-stock flag — parsing markdown back into fields is just scraping again. The real power of a managed API is the **`json` format**: you hand Firecrawl a **schema** plus a natural-language prompt, and its LLM fills the schema straight from the page. **No selectors, resilient to layout changes** — the thing DIY scraping (NB 47) can't easily match.

The production schema type is a **`pydantic` model** (validation + JSON-schema for free). We wrap it in a `try/except` so the notebook still runs where `pydantic` isn't installed, falling back to a stdlib **`dataclass`** — the exact pattern from **NB 25**'s structured extraction.

In [6]:
# The schema Firecrawl fills. Pydantic is the production choice; the dataclass
# fallback keeps this cell runnable everywhere (same pattern as NB 25).
try:
    from pydantic import BaseModel, Field

    class Product(BaseModel):
        title: str = Field(description="the book's title")
        price: float = Field(description="price in GBP")
        rating: int = Field(description="stars, 1-5")
        in_stock: bool = Field(description="is it available")

    SCHEMA_ENGINE = "pydantic"
except ModuleNotFoundError:
    from dataclasses import dataclass

    @dataclass
    class Product:
        title: str
        price: float
        rating: int
        in_stock: bool

    SCHEMA_ENGINE = "dataclass (pydantic not installed)"

print("schema engine:", SCHEMA_ENGINE)

schema engine: dataclass (pydantic not installed)


In [7]:
# Ask for typed JSON: pass the schema + a prompt inside a {"type": "json", ...} format.
resp = app.scrape(
    "https://shop.example.com/catalogue/page-1",
    formats=[{"type": "json", "schema": Product, "prompt": "Extract every book with price, rating, and stock."}],
)
raw = resp.json["products"]              # the LLM-filled records, as plain dicts
print("raw record from Firecrawl:", raw[0])

# Validate at the boundary (NB 25's habit): turn dicts into typed objects...
products = [Product(**r) for r in raw]

# ...and normalise back to rows. Pydantic has .model_dump(); a dataclass uses vars().
def to_row(obj):
    return obj.model_dump() if hasattr(obj, "model_dump") else vars(obj)

import pandas as pd
products_df = pd.DataFrame([to_row(p) for p in products])
print(f"\nvalidated {len(products_df)} products with {SCHEMA_ENGINE}:")
print(products_df.to_string(index=False))

raw record from Firecrawl: {'title': 'Clean Code', 'price': 32.5, 'rating': 4, 'in_stock': True}

validated 3 products with dataclass (pydantic not installed):
                                title  price  rating  in_stock
                           Clean Code  32.50       4      True
             The Pragmatic Programmer  28.99       5      True
Designing Data-Intensive Applications  41.00       5     False


> 🧠 **Why the schema, not the markdown?** Two identical shops with wildly different HTML produce the **same typed records** — the schema is the contract, the page's markup is an implementation detail. That is the whole promise: you stop maintaining selectors and start declaring *what you want*. The validated `products_df` is now ready to load into SQL (NB 13) or embed for retrieval (Module 6, NB 23) — scraping is an **ingestion** step, not the product.

---

### ✋ Quick exercise (~2 min) — Summarise the extracted catalogue

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using the validated `products_df` from §4 (no re-scraping), keep only the **in-stock** books and compute their **average price**. A one-line ingestion summary you'd log after every extraction.

In [8]:
# ✍️ Your turn 👇
# Reuse `products_df` from §4 — no re-scraping. Don't mutate it.
in_stock_df = ...     # keep only rows where in_stock is True
avg_price   = ...     # the mean `price` of those rows
# print(f"{len(in_stock_df)} in stock · avg £{avg_price:.2f}")   # ← uncomment once filled in

<details>
<summary>✅ <b>Solution</b></summary>

```python
in_stock_df = products_df[products_df["in_stock"]]
avg_price   = in_stock_df["price"].mean()
print(f"{len(in_stock_df)} in stock · avg £{avg_price:.2f}")
```

`products_df["in_stock"]` is a boolean mask that keeps only available titles before averaging — the same "filter then aggregate" move you'd run on any freshly ingested table.
</details>

## 5. `crawl`, `map`, and `search` — beyond a single page

One URL is rarely the whole job. Three more verbs cover a site:

- **`crawl(url, limit=...)`** follows links across a site and returns **content for every page**. Because a crawl can be large, it comes back as a **job dict** — `{"status", "total", "completed", "data": [...documents...]}` — *not* a single document. (Remember that shape; it's a classic gotcha — see the Debug-me exercise.)
- **`map(url)`** returns **just the list of URLs**, fast. Perfect for planning a crawl or feeding a sitemap.
- **`search(query)`** runs a **web search and scrapes the results** in one call — great for "find the docs page about X and give me its markdown."

In [9]:
# CRAWL — follow links across the site. Note: the result is a JOB DICT; the documents
# live under ["data"], each with its own .markdown and .metadata.
crawl_result = app.crawl("https://shop.example.com", limit=5)
print(f"crawl status: {crawl_result['status']} · {crawl_result['total']} pages\n")
for page in crawl_result["data"]:
    first_line = page.markdown.splitlines()[0]
    print(f"  {page.metadata['sourceURL']:<45} {first_line}")

crawl status: completed · 4 pages

  https://shop.example.com/                     # The Polite Bookshop
  https://shop.example.com/catalogue/page-1     # The Polite Bookshop — Catalogue (page 1)
  https://shop.example.com/catalogue/page-2     # Catalogue — page 2
  https://shop.example.com/about                # About


In [10]:
# MAP — just the URLs on the site (fast; plan a crawl from this).
mapped = app.map("https://shop.example.com")
print("mapped URLs:")
for u in mapped["links"]:
    print("  ", u)

# SEARCH — web search + scrape the hits in one call.
hits = app.search("data-intensive applications book", limit=2)
print("\nsearch hits:")
for h in hits["data"]:
    print(f"  {h['title']:<42} → {h['url']}")

mapped URLs:
   https://shop.example.com/
   https://shop.example.com/catalogue/page-1
   https://shop.example.com/catalogue/page-2
   https://shop.example.com/products/clean-code
   https://shop.example.com/products/pragmatic-programmer
   https://shop.example.com/products/ddia
   https://shop.example.com/about

search hits:
  Designing Data-Intensive Applications      → https://shop.example.com/products/ddia
  DDIA — a review                            → https://other.example.com/reviews/ddia


---

### ✋ Quick exercise (~2 min) — List the crawled URLs

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Reusing `crawl_result` from §5 (no re-crawling), build a **list of every page's `sourceURL`**. Remember: the documents are under `crawl_result["data"]`, and each one keeps its URL in `.metadata["sourceURL"]`.

In [11]:
# ✍️ Your turn 👇
# Reuse `crawl_result` from §5 — no re-crawling.
source_urls = ...     # every crawled page's sourceURL (from each document's .metadata)
# print(len(source_urls), "pages:", source_urls)   # ← uncomment once filled in

<details>
<summary>✅ <b>Solution</b></summary>

```python
source_urls = [page.metadata["sourceURL"] for page in crawl_result["data"]]
print(f"{len(source_urls)} pages crawled:")
for u in source_urls:
    print("  ", u)
```

The trap is reaching for `crawl_result["markdown"]` — but a crawl returns a **job**, so you first index into `["data"]` to get the list of documents, *then* read each document. This is the single most common Firecrawl bug (you'll meet it again in the Debug-me exercise).
</details>

## 6. Batching many URLs into a DataFrame

The everyday pipeline is: **a list of URLs → typed records → one DataFrame**. Loop over the URLs, ask each for the `json` format, and collect the records. Here each product-detail page returns a single richer record (it even has an `author` the list view didn't), plus we stamp the `source_url` so every row is traceable back to its page.

In [12]:
# Batch: scrape a list of product-detail URLs, each into one typed record, then tabulate.
PRODUCT_URLS = [
    "https://shop.example.com/products/clean-code",
    "https://shop.example.com/products/pragmatic-programmer",
    "https://shop.example.com/products/ddia",
]

rows = []
for url in PRODUCT_URLS:
    d = app.scrape(url, formats=[{"type": "json", "schema": Product, "prompt": "Extract the product."}])
    rec = d.json["product"]                    # a single record on a detail page
    rows.append({**rec, "source_url": url})    # keep provenance: which page each row came from

catalog_df = pd.DataFrame(rows)
print(catalog_df.to_string(index=False))
print(f"\n{len(catalog_df)} products · {catalog_df['in_stock'].sum()} in stock · avg £{catalog_df['price'].mean():.2f}")

                                title  price  rating  in_stock           author                                             source_url
                           Clean Code  32.50       4      True Robert C. Martin           https://shop.example.com/products/clean-code
             The Pragmatic Programmer  28.99       5      True    Hunt & Thomas https://shop.example.com/products/pragmatic-programmer
Designing Data-Intensive Applications  41.00       5     False Martin Kleppmann                 https://shop.example.com/products/ddia

3 products · 2 in stock · avg £34.16


---

### ✋ Quick exercise (~2 min) — Flag the premium titles

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using `catalog_df` from §6 (don't mutate it), build a boolean mask for **premium** books — those priced **above £35** — and count how many there are.

In [13]:
# ✍️ Your turn 👇
# Reuse `catalog_df` from §6 — don't mutate it.
premium_mask = ...     # a boolean Series: is `price` above 35?
n_premium    = ...     # how many titles is that?
# print(n_premium, "premium titles (> £35)")   # ← uncomment once filled in

<details>
<summary>✅ <b>Solution</b></summary>

```python
premium_mask = catalog_df["price"] > 35
n_premium = int(premium_mask.sum())
print(catalog_df[premium_mask][["title", "price"]].to_string(index=False))
print(n_premium, "premium titles (> £35)")
```

`catalog_df["price"] > 35` returns a boolean Series you can both count (`.sum()`) and use to filter (`catalog_df[premium_mask]`) — the bread-and-butter of turning scraped records into an answer.
</details>

## 7. Cost, rate limits, and when *not* to reach for it

A managed API is not free money. Every `scrape` is a **billed credit** and a **network round-trip**; a `crawl` can be hundreds of them. Treat it like any paid dependency:

- ⚠️ **Cost scales with pages.** `crawl` on a big site can burn your monthly credits in one run — always set a `limit`, and prefer `map` (cheap) to plan before you `crawl` (expensive).
- ⚠️ **Rate limits are real.** Batch loops need throttling and retry-with-backoff (the polite-client habits from NB 47 and NB 12), even though the *anti-bot* part is now the API's problem, not yours.
- ⚠️ **Cache aggressively.** The same "never fetch the same page twice" rule from NB 47 saves both credits and latency — memoise by URL (see the Stretch exercises).
- ⚠️ **It's still someone's data.** A managed API does **not** absolve you of `robots.txt`, Terms of Service, copyright, or PII/GDPR duties. The ethics of NB 47 travel with you.

**So when is DIY still the right call?** Here's the decision table against NB 47's hand-rolled stack:

| Situation | Reach for… |
|---|---|
| Official **API / data export / RSS** exists | that (NB 12) — never scrape what you can request |
| **Static, well-structured** HTML, low volume, you control the selectors | **`requests` + BeautifulSoup** (NB 47) — 3 lines, free |
| **JavaScript-rendered** content, but staying in-house | a headless browser (Playwright / Selenium) |
| **JS + anti-bot + many pages**, or you just want **clean markdown** | **Firecrawl** (this notebook) |
| **Typed records** out of messy pages, no selectors to maintain | **Firecrawl** `json` extraction (§4) |
| Feeding a **RAG** index or an **agent** | **Firecrawl** markdown → chunk → embed (Module 6, NB 23) |

> 🧱 **The one-liner.** If a static page would fall to `requests` + BeautifulSoup in three lines, do *that*. Reach for Firecrawl when the page fights back — JavaScript, anti-bot, scale, or "just give me clean text for my LLM." Paying to delete a class of problems is worth it; paying to re-implement `requests.get` is not.

## 🧪 Practice exercises

Work these in the empty cells (or the buggy one). Solutions are collapsed — try first, then peek.

### Exercise 1 — ⭐ Grab the outbound links

Scrape `"https://shop.example.com/catalogue/page-1"` asking for **both** `"markdown"` and `"links"`, then print how many links came back and list them. (Hint: the `links` format populates `doc.links`.)

In [14]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
doc_links = app.scrape("https://shop.example.com/catalogue/page-1", formats=["markdown", "links"])
print(len(doc_links.links), "links found:")
for u in doc_links.links:
    print("  ", u)
```

Asking for several formats at once is free-form: you get the markdown *and* the link list on the same document.
</details>

### Exercise 2 — ⭐⭐ Design a schema for a different page

You want to scrape **conference talks** into a table. Define a schema for a talk — `title`, `speaker`, `track`, and `start` (ISO time string) — reusing the `try/except` pydantic-or-dataclass pattern from §4, then validate a couple of mock records against it and print them.

In [15]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
try:
    from pydantic import BaseModel
    class Talk(BaseModel):
        title: str
        speaker: str
        track: str
        start: str        # ISO time string; use datetime in production
except ModuleNotFoundError:
    from dataclasses import dataclass
    @dataclass
    class Talk:
        title: str
        speaker: str
        track: str
        start: str

# What Firecrawl's LLM extraction would return for formats=[{"type":"json","schema":Talk,...}]:
extracted = [
    {"title": "Conformal Prediction in Practice", "speaker": "A. Researcher", "track": "ML",   "start": "2026-09-01T09:00"},
    {"title": "Scraping the Modern Web",          "speaker": "B. Engineer",   "track": "Data", "start": "2026-09-01T10:30"},
]
talks = [Talk(**t) for t in extracted]
for t in talks:
    print(f"  {t.start}  [{t.track}]  {t.title} — {t.speaker}")
```

The schema *is* the interface: change the page, the fields you declared stay the same.
</details>

### Exercise 3 — ⭐⭐ Map, then keep only the product pages

Call `app.map("https://shop.example.com")`, then filter the returned URLs down to just the **product-detail** pages (the ones with `/products/` in the path). Print the filtered list.

In [16]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
urls = app.map("https://shop.example.com")["links"]
product_urls = [u for u in urls if "/products/" in u]
print(f"{len(product_urls)} product pages of {len(urls)} total:")
for u in product_urls:
    print("  ", u)
```

`map` is the cheap "what's on this site?" call — filtering its output is how you build the exact URL list to feed a batched scrape (§6) without paying for a full crawl.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The cell below wants the markdown of the first crawled page. Run it, predict the error, then fix it. (Reminder from §5: what *shape* does `crawl()` return?)

In [17]:
# ⚠️ THIS CELL INTENTIONALLY ERRORS — read the prompt above, predict the failure, then fix it.
job = app.crawl("https://shop.example.com", limit=3)
first_markdown = job["markdown"]          # BUG: crawl() returns a job dict, not a document
print(first_markdown[:60])

KeyError: 'markdown'

<details>
<summary>💡 <b>Solution</b></summary>

`crawl()` returns a **job dict** — `{"status", "total", "completed", "data": [...]}` — so `job["markdown"]` raises `KeyError: 'markdown'`. There is no top-level markdown; the per-page documents live under `["data"]`, and *each* document has a `.markdown`:

```python
job = app.crawl("https://shop.example.com", limit=3)
first_markdown = job["data"][0].markdown     # index into ["data"], then read the document
print(first_markdown[:60])
```

The mental model: `scrape` → one document (`.markdown`), but `crawl` → a job wrapping *many* documents (`["data"][i].markdown`). Mixing up the two shapes is the #1 Firecrawl beginner bug.
</details>

## 🧠 Stretch exercises

Harder, open-ended. Solutions sketch one good approach — yours may differ.

### Stretch exercise A — ⭐⭐⭐ A credit-saving cache

Every `scrape` costs a credit. Write a thin `CachedFirecrawl` wrapper that memoises `scrape` results by `(url, tuple-of-format-keys)` so a repeated call returns instantly without hitting the client again. Prove it: scrape the same URL twice and show the second call was a cache hit.

<details>
<summary>💡 <b>Solution</b></summary>

```python
class CachedFirecrawl:
    def __init__(self, client):
        self.client = client
        self._cache = {}
        self.hits = 0

    def scrape(self, url, formats=None):
        # Build a hashable key from the URL + a stable signature of the formats asked for.
        sig = tuple(f.get("type", "markdown") if isinstance(f, dict) else f for f in (formats or ["markdown"]))
        key = (url, sig)
        if key in self._cache:
            self.hits += 1
            return self._cache[key]
        doc = self.client.scrape(url, formats=formats)
        self._cache[key] = doc
        return doc

cached = CachedFirecrawl(app)
a = cached.scrape("https://shop.example.com/catalogue/page-1", formats=["markdown"])
b = cached.scrape("https://shop.example.com/catalogue/page-1", formats=["markdown"])  # cache hit
print("same object returned:", a is b, "· cache hits:", cached.hits)
```

This is the "cache, don't re-fetch" rule from NB 47 applied to a *paid* API — the savings are now money, not just politeness.
</details>

### Stretch exercise B — ⭐⭐⭐ A richer schema with optional fields

Extend §4's extraction with an **optional** field and a **list** field: add `author` (may be missing) and `tags: list[str]`. Make it work in *both* the pydantic and dataclass branches (hint: `Optional[str] = None` / `field(default_factory=list)`), then validate a record that omits `author`.

<details>
<summary>💡 <b>Solution</b></summary>

```python
from typing import Optional, List
try:
    from pydantic import BaseModel
    class RichProduct(BaseModel):
        title: str
        price: float
        author: Optional[str] = None
        tags: List[str] = []
except ModuleNotFoundError:
    from dataclasses import dataclass, field
    @dataclass
    class RichProduct:
        title: str
        price: float
        author: Optional[str] = None
        tags: List[str] = field(default_factory=list)

p = RichProduct(**{"title": "Refactoring", "price": 34.0, "tags": ["engineering", "classics"]})  # no author
print(p)
```

Optional + default-list fields are how you stay robust to pages that don't always fill every field — the LLM extractor simply leaves them at their default.
</details>

### Stretch exercise C — ⭐⭐⭐ Turn a crawl into a RAG-ready table

Take a `crawl_result` and build a DataFrame with one row per page: `url`, `markdown`, and `n_chars` (length of the markdown). This is the exact hand-off into a retrieval pipeline — chunk + embed each row's markdown (Module 6, NB 23).

<details>
<summary>💡 <b>Solution</b></summary>

```python
crawl_result = app.crawl("https://shop.example.com", limit=5)
rag_df = pd.DataFrame([
    {"url": p.metadata["sourceURL"], "markdown": p.markdown, "n_chars": len(p.markdown)}
    for p in crawl_result["data"]
])
print(rag_df[["url", "n_chars"]].to_string(index=False))
print(f"\n{len(rag_df)} pages · {rag_df['n_chars'].sum()} chars total → chunk + embed next (NB 23).")
```

Scraping ends where retrieval begins: a `(url, markdown)` table is precisely what a chunker consumes.
</details>

## 🎁 Bonus mini-project — A schema-driven catalogue extractor

Tie it all together into one reusable function:

1. Write `extract_catalog(urls, client=app) -> pd.DataFrame` that scrapes each URL in `urls` with the `json` format + the `Product` schema.
2. Validate every returned record through `Product(**rec)` and normalise with `to_row` (§4).
3. Add a `source_url` column so each row is traceable, and skip (with a warning) any page whose `.json` is missing.
4. Return the tidy DataFrame, then run it over the three `/products/...` URLs and print a one-line summary.

The result is a tiny, testable ingestion component you could drop into a scheduled job (NB 40).

In [18]:
# Your code here  👇


<details>
<summary>💡 <b>Solution sketch</b></summary>

```python
def extract_catalog(urls, client=app):
    rows = []
    for url in urls:
        doc = client.scrape(url, formats=[{"type": "json", "schema": Product, "prompt": "Extract the product."}])
        data = doc.json or {}
        rec = data.get("product") or (data.get("products") or [None])[0]
        if rec is None:
            print(f"  ⚠️  no json for {url} — skipping")
            continue
        product = Product(**{k: rec[k] for k in ("title", "price", "rating", "in_stock")})
        rows.append({**to_row(product), "source_url": url})
    return pd.DataFrame(rows)

catalog = extract_catalog(PRODUCT_URLS)
print(catalog.to_string(index=False))
print(f"\n{len(catalog)} products extracted · avg £{catalog['price'].mean():.2f}")
```

Notice the defensive `data.get(...)` and the skip-with-warning: real extraction sometimes returns nothing for a page, and a batch job must survive one bad URL. Validating through `Product(**...)` guarantees every row that *does* make it in is well-typed — ready for SQL (NB 13) or a scheduled run (NB 40).
</details>

## 🧠 Key takeaways

- **Managed scraping earns its keep when DIY (NB 47) breaks down:** JavaScript rendering, anti-bot defences, crawling at scale, or "just give me clean text." For a static page, `requests` + BeautifulSoup in three lines still wins.
- **Firecrawl turns a URL into LLM-ready content** — clean **markdown** or schema-validated **typed JSON** — via four verbs: `scrape` (one page), `crawl` (a site), `map` (its URLs), `search` (the web).
- **`scrape` → one document** with `.markdown`, `.metadata`, `.links`, `.json`. **`crawl` → a job dict** whose documents live under `["data"]` — mixing up those two shapes is the #1 beginner bug.
- **Typed JSON beats markdown for pipelines:** a **schema** (pydantic, or a dataclass fallback — the NB 25 pattern) means you declare *what you want* instead of maintaining brittle selectors. Validate at the boundary.
- **Batch URLs → a DataFrame** and you have a RAG-ready / SQL-ready ingestion step — scraping is the *pipe that fills the product*, not the product.
- **It still costs money and still has duties:** set `limit`s, cache by URL, throttle your batches, and honour `robots.txt` / ToS / PII exactly as in NB 47.

## ✅ Self-assessment

- [ ] Explain three DIY pain points that push you toward a managed scraping API
- [ ] Bootstrap a Firecrawl client that falls back to a mock with no key or network
- [ ] Scrape one URL into markdown and read its `.metadata`
- [ ] Extract typed JSON with a schema and validate it (pydantic *or* dataclass)
- [ ] Correctly read a `crawl` result (`["data"][i].markdown`, not `["markdown"]`)
- [ ] Batch a list of URLs into a tidy DataFrame with provenance
- [ ] Decide when DIY scraping (NB 47) is still the right call

## 🚀 Next step

You can now turn *any* URL into clean, typed data. The rest of Module 16 puts that skill to work on real sources.

Continue with **Notebook 49 — OpenAlex & Scholarly Data** (`49_openalex_scholarly_data.ipynb`): a clean, open **API** for millions of papers — the "use the official API before you scrape" principle (NB 12) applied to research data, and a natural companion to the scraping toolkit you just built.

> 🚀 Change one URL. Swap the schema. Re-run. The pipeline is the same; only the target moves.